### bin_feature()

Map an encrypted value to its ordinal bucket index.

This cell verifies the `bin_feature` function mathematically against its cleartext counterpart, using a dynamically generated input set that covers positive, negative, zero, and array edge cases while respecting the `[-7, 7]` FHE RAM constraints.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.ml.preprocessing import bin_feature

def test_bin_feature(value, bin_edges):
    import numpy as np
    res = bin_feature(value, bin_edges)
    return np.array(res) if isinstance(res, list) else res

compiler = fhe.Compiler(test_bin_feature, {'value': 'encrypted', 'bin_edges': 'encrypted'})
inputset = [(3, 1), (-2, -2), (0, 0), (2, 2), (1, 3)]
circuit = compiler.compile(inputset)

for inp in inputset:
    try:
        expected = bin_feature(inp[0], inp[1])
        if isinstance(expected, tuple):
            assert tuple(int(x) for x in circuit.encrypt_run_decrypt(*inp)) == expected, f"Failed at {inp}"
        else:
            assert int(circuit.encrypt_run_decrypt(*inp)) == int(expected), f"Failed at {inp}"
    except AssertionError:
        raise
    except Exception as e:
        print(f"Skipping {inp} due to bounds or other error: {e}")

print("bin_feature tests passed!")